In [1]:
%%time
import pandas as pd
import numpy
import os
from collections import Counter
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger, CkipNerChunker


/Users/xx/miniforge3/envs/ai23/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 763 ms, sys: 153 ms, total: 917 ms
Wall time: 1.3 s


In [6]:
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger, CkipNerChunker
import pandas as pd
from collections import Counter

file_path = os.path.join('..','1.爬蟲', 'jobs.csv')
df = pd.read_csv(file_path, sep='|')
# 初始化 CKIP 模型
ws = CkipWordSegmenter(model="albert-tiny") 
pos = CkipPosTagger(model="albert-tiny")
ner = CkipNerChunker(model="albert-tiny")

# 過濾空值，並轉成字串
df = df[df['jobDetail'].notna() & df['welfare'].notna()]
df['jobDetail'] = df['jobDetail'].astype(str)
df['welfare'] = df['welfare'].astype(str)

# CKIP 斷詞
tokens_detail = ws(df['jobDetail'].tolist())
tokens_welfare = ws(df['welfare'].tolist())

# 詞性標註
tokens_detail_pos = pos(tokens_detail)
tokens_welfare_pos = pos(tokens_welfare)

# 詞性對應配對
word_pos_detail = [list(zip(w, p)) for w, p in zip(tokens_detail, tokens_detail_pos)]
word_pos_welfare = [list(zip(w, p)) for w, p in zip(tokens_welfare, tokens_welfare_pos)]

# 命名實體辨識
entities_detail = ner(df['jobDetail'])
entities_welfare = ner(df['welfare'])

# 過濾條件: 兩個字以上 + 指定詞性
allowPOS = ['Na', 'Nb', 'Nc', 'Nv', 'Vc']

def filter_tokens(word_pos_pair):
    return [[w for w, p in wp if len(w) >= 2 and p in allowPOS] for wp in word_pos_pair]

tokens_detail_v2 = filter_tokens(word_pos_detail)
tokens_welfare_v2 = filter_tokens(word_pos_welfare)

# 字頻統計函式
def word_frequency(wp_pair):
    filtered_words = []
    for word, pos in wp_pair:
        if (pos in allowPOS) & (len(word) >= 2):
            filtered_words.append(word)
    return Counter(filtered_words).most_common(200)

# 建立字頻資料
keyfreq_detail = [word_frequency(wp) for wp in word_pos_detail]
keyfreq_welfare = [word_frequency(wp) for wp in word_pos_welfare]

# 寫入 DataFrame
df['tokens'] = tokens_detail
df['tokens_v2'] = tokens_detail_v2
df['token_pos'] = word_pos_detail
df['entities'] = entities_detail
df['top_key_freq'] = keyfreq_detail

df['welfare_tokens'] = tokens_welfare
df['welfare_tokens_v2'] = tokens_welfare_v2
df['welfare_token_pos'] = word_pos_welfare
df['welfare_entities'] = entities_welfare
df['welfare_top_key_freq'] = keyfreq_welfare

# 欄位順序整理
df = df[[
    'jobType', 'header', 'jobDetail', 'salary', 'workExp', 'edu', 'tool', 'welfare', 'area', 'date', 'link','final_salary',
    'top_key_freq', 'tokens', 'tokens_v2', 'token_pos', 'entities',
    'welfare_top_key_freq', 'welfare_tokens', 'welfare_tokens_v2', 'welfare_token_pos', 'welfare_entities'
]]

# 儲存檔案
df.to_csv('104_preprocessed.csv', sep='|', index=False)

print("Tokenization for jobDetail and welfare done!")

Inference: 100%|██████████| 5/5 [00:17<00:00,  3.59s/it]


Tokenization for jobDetail and welfare done!
